## Mode Selection

In [ ]:
import os, sys
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="strict")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="strict")

# Mode A: Train LoRA from scratch on Kaggle GPU
TRAIN_ON_KAGGLE = 1

# Mode B: Use pre-trained LoRA weights from dataset and just package them
USE_PRETRAINED = 0

assert (TRAIN_ON_KAGGLE + USE_PRETRAINED) == 1, \
    "Set exactly one of TRAIN_ON_KAGGLE / USE_PRETRAINED to 1."

PRETRAINED_ADAPTER_DATASET_PATH = "/kaggle/input/datasets/konbu17/nemotron-sft-lora-cot-selection"
BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

print({
    "TRAIN_ON_KAGGLE": TRAIN_ON_KAGGLE,
    "USE_PRETRAINED": USE_PRETRAINED,
    "PRETRAINED_ADAPTER_DATASET_PATH": PRETRAINED_ADAPTER_DATASET_PATH,
})

## Setup & Model Loading

In [ ]:
import os, glob, sys, subprocess, site

candidates = glob.glob("/kaggle/input/**/*triton*.whl", recursive=True)
print("Found Triton wheels:", candidates)

if not candidates:
    raise FileNotFoundError("No Triton wheel found under /kaggle/input")
wheel = candidates[0]

target = "/kaggle/working/pydeps"
os.makedirs(target, exist_ok=True)

subprocess.run(
    [
        sys.executable, "-m", "pip", "install",
        "--no-deps", "--target", target,
        "--upgrade", "--ignore-installed", wheel,
    ],
    check=True,
)

if target not in sys.path:
    sys.path.insert(0, target)
site.addsitedir(target)

import importlib.util
print("triton spec:", importlib.util.find_spec("triton"))

In [ ]:
if TRAIN_ON_KAGGLE:
    import sys, os, shutil, stat

    sys.path.insert(0, '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script')

    ptxas_src = '/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell'
    ptxas_dst = '/tmp/ptxas-blackwell'
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        src_bin = os.path.dirname(ptxas_src)
        dst_bin = '/tmp/triton_nvidia_bin'
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

        os.environ['TRITON_PTXAS_BLACKWELL_PATH'] = ptxas_dst

        import triton.backends.nvidia as nv_backend
        nv_backend.__file__ = os.path.join(dst_bin, '..', '__init__.py')
        os.environ['TRITON_PTXAS_PATH'] = ptxas_dst

    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: '12.0'

    print('Training environment fixes applied.')
else:
    print("USE_PRETRAINED=1: skipping Triton / ptxas environment fixes.")

In [ ]:
if TRAIN_ON_KAGGLE:
    print("Skip standalone trl install; Unsloth setup cell installs compatible packages.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import glob, os, subprocess, sys

    def recursive_wheels(pattern: str):
        return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))

    packages_dir = "/kaggle/input/datasets/mayukh18/nemotron-packages/packages"
    all_mamba  = recursive_wheels("mamba_ssm-*.whl")
    all_causal = recursive_wheels("causal*conv1d*.whl")

    import torch
    print("Python:", sys.version)
    print("Torch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())

    if not torch.cuda.is_available():
        raise RuntimeError("TRAIN_ON_KAGGLE=1 requires a GPU runtime.")
    if not os.path.isdir(packages_dir):
        raise FileNotFoundError(f"Offline wheel directory not found: {packages_dir}")

    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q",
            "--no-index", "--find-links", packages_dir,
            "unsloth", "trl", "peft", "transformers",
            "datasets", "accelerate", "bitsandbytes",
        ],
        check=True,
    )

    def pick_last(wheels): return wheels[-1] if wheels else None

    causal_wheel = pick_last(all_causal)
    mamba_wheel  = pick_last(all_mamba)
    if causal_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal_wheel], check=True)
    if mamba_wheel:
        subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba_wheel], check=True)
    else:
        raise FileNotFoundError("No compatible mamba_ssm wheel found under /kaggle/input.")

    print("Offline package installation finished.")

In [ ]:
if TRAIN_ON_KAGGLE:
    import torch
    import kagglehub
    from unsloth import FastLanguageModel

    MAX_SEQ_LEN = 8192
    MODEL_PATH = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    print(f"Model path: {MODEL_PATH}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_PATH,
        max_seq_length=MAX_SEQ_LEN,
        load_in_4bit=False,
        load_in_8bit=False,
        full_finetuning=False,
        trust_remote_code=True,
        unsloth_force_compile=False,
        attn_implementation="eager",
        dtype=torch.bfloat16,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"  # SFT uses right-padding for causal LM loss
    print("Model loaded with Unsloth.")
else:
    print("USE_PRETRAINED=1: skipping base model and tokenizer loading.")

## IGU-LoRA Configuration

In [ ]:
if TRAIN_ON_KAGGLE:
    from unsloth import FastLanguageModel

    # ============================================================
    # IGU-LoRA CONFIG — plain SFT, maximum coverage
    # ============================================================
    # IGU = Improved GPU Utilization:
    #   - RSLoRA normalizes gradients by sqrt(r), stable at high rank
    #   - Rank 32 (competition max) on all projections
    #   - Alpha 64 = 2× rank gives strong adapter influence
    #   - Dropout 0.0: RSLoRA provides implicit regularization
    #   - lm_head EXCLUDED: its gradient drifts \boxed{} token
    #     distribution and hurts format consistency at inference
    # ============================================================
    LORA_RANK    = 32
    LORA_ALPHA   = 64
    LORA_DROPOUT = 0.0

    target_modules = [
        # Attention projections
        "q_proj", "k_proj", "v_proj", "o_proj",
        "in_proj", "out_proj",
        # MLP / MoE feed-forward
        "gate_proj", "up_proj", "down_proj",
        # Mamba SSM projections (Nemotron hybrid architecture)
        "x_proj", "dt_proj",
        # lm_head intentionally EXCLUDED
    ]

    print(f"IGU-LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
    print(f"Targets: {target_modules}")

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_RANK,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=target_modules,
        bias="none",
        use_gradient_checkpointing=True,
        random_state=42,
        use_rslora=True,
        use_dora=False,
    )
    model.print_trainable_parameters()
else:
    print("USE_PRETRAINED=1: skipping LoRA construction.")

## SFT Training

In [ ]:
# ============================================================
# MEMORY OPTIMIZATIONS
# ============================================================
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"

import gc
import re
import subprocess
import time

import pandas as pd
import torch
from datasets import Dataset as HFDataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

# ============================================================
# GPU METRICS CALLBACK (TensorBoard)
# ============================================================
class GPUMetricsCallback(TrainerCallback):
    def __init__(self, log_every_n_steps=2):
        super().__init__()
        self.log_every_n_steps = log_every_n_steps
        self._last_step_time   = None
        self._last_global_step = 0

    def _query_nvidia_smi(self):
        try:
            r = subprocess.run(
                ["nvidia-smi",
                 "--query-gpu=utilization.gpu,memory.used,memory.total,temperature.gpu,power.draw",
                 "--format=csv,noheader,nounits"],
                capture_output=True, text=True, timeout=5)
            if r.returncode != 0:
                return None
            p = [x.strip() for x in r.stdout.strip().split("\n")[0].split(",")]
            return {
                "gpu/utilization_percent": float(p[0]),
                "gpu/memory_used_gb":      float(p[1]) / 1024.0,
                "gpu/temperature_celsius": float(p[3]),
                "gpu/power_watts":         float(p[4]) if p[4] != "[N/A]" else 0.0,
            }
        except Exception:
            return None

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is None or state.global_step % self.log_every_n_steps != 0:
            return
        smi = self._query_nvidia_smi()
        if smi:
            logs.update(smi)
        if torch.cuda.is_available():
            logs["gpu/memory_allocated_gb"]     = torch.cuda.memory_allocated()     / (1024**3)
            logs["gpu/memory_reserved_gb"]      = torch.cuda.memory_reserved()      / (1024**3)
            logs["gpu/max_memory_allocated_gb"] = torch.cuda.max_memory_allocated() / (1024**3)
        now = time.time()
        if self._last_step_time is not None:
            elapsed = now - self._last_step_time
            steps   = state.global_step - self._last_global_step
            if elapsed > 0 and steps > 0:
                sps = steps / elapsed
                logs["throughput/steps_per_sec"]   = sps
                logs["throughput/samples_per_sec"] = sps * args.per_device_train_batch_size
        self._last_step_time   = now
        self._last_global_step = state.global_step

    def on_train_begin(self, args, state, control, **kwargs):
        self._last_step_time   = time.time()
        self._last_global_step = state.global_step
        print("[TensorBoard] GPU metrics logging enabled.")

    def on_train_end(self, args, state, control, **kwargs):
        smi = self._query_nvidia_smi()
        if smi:
            peak = torch.cuda.max_memory_allocated() / (1024**3)
            print(f"[TensorBoard] Done. Peak mem: {peak:.1f} GB | Temp: {smi['gpu/temperature_celsius']:.0f}C")

print("Imports and GPUMetricsCallback ready.")

In [ ]:
if TRAIN_ON_KAGGLE:
    # ============================================================
    # DATA LOADING
    # ============================================================
    SEED = 42
    PROMPT_SUFFIX = "\nPlease put your final answer inside `\\boxed{}`. For example: `\\boxed{your answer}`"

    DATASET_PATH = "/kaggle/input/datasets/dgxchen/nemotron-cot-tong/problem_ids_matched.csv"
    df = pd.read_csv(DATASET_PATH)
    print(f"Full dataset: {len(df)} rows")

    df = df.dropna(subset=["prompt", "answer", "generated_cot"]).reset_index(drop=True)
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    print(f"After filter: {len(df)} rows")

    records, skipped = [], 0
    for _, row in df.iterrows():
        prompt = str(row["prompt"])
        answer = str(row["answer"])
        cot    = str(row["generated_cot"])
        if cot == "nan" or len(cot.strip()) < 5 or len(cot.strip()) > 8100:
            skipped += 1
            continue
        # Strip any \boxed{} the CoT generator leaked in — model learns to produce it
        cot_clean = re.sub(r'\\boxed\{[^}]*\}', '', cot).rstrip()
        records.append({
            "messages": [
                {"role": "user",      "content": prompt + PROMPT_SUFFIX},
                {"role": "assistant", "content": f"<think>\n{cot_clean}\n</think>\n\\boxed{{{answer}}}"},
            ]
        })

    dataset = HFDataset.from_list(records)
    print(f"SFT records: {len(records)} (skipped {skipped} invalid CoT)")

    def formatting_prompts_func(example):
        messages = example["messages"]
        if messages and isinstance(messages[0], dict):
            conversations = [messages]
        else:
            conversations = messages
        texts = []
        for convo in conversations:
            try:
                text = tokenizer.apply_chat_template(
                    convo, tokenize=False,
                    add_generation_prompt=False, enable_thinking=True)
            except TypeError:
                text = tokenizer.apply_chat_template(
                    convo, tokenize=False, add_generation_prompt=False)
            texts.append(text)
        return texts

    # ============================================================
    # TRAINING CONFIG — IGU-SFT v9.2
    # ============================================================
    TB_LOG_DIR = "/kaggle/working/tb_logs"

    training_args = SFTConfig(
        output_dir="/kaggle/working/sft_output",
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,      # effective batch = 8
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        max_length=8192,
        optim="adamw_8bit",
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        weight_decay=0.005,
        max_grad_norm=0.5,
        neftune_noise_alpha=5.0,
        logging_steps=2,
        logging_dir=TB_LOG_DIR,
        report_to="tensorboard",
        save_strategy="no",
        bf16=True,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        dataloader_num_workers=2,
        remove_unused_columns=False,
        seed=SEED,
        packing=False,
        dataset_num_proc=4,
    )

    print("\n" + "=" * 60)
    print("  IGU-SFT TRAINING CONFIG v9.2")
    print("=" * 60)
    print(f"  LR:        {training_args.learning_rate}")
    print(f"  Epochs:    {training_args.num_train_epochs}")
    print(f"  Warmup:    {training_args.warmup_ratio}")
    print(f"  NEFTune:   {training_args.neftune_noise_alpha}")
    bs = training_args.per_device_train_batch_size
    ga = training_args.gradient_accumulation_steps
    print(f"  Batch:     {bs} x {ga} = {bs * ga}")
    print(f"  Optimizer: {training_args.optim}")
    print(f"  TBoard:    {TB_LOG_DIR}")
    print("=" * 60 + "\n")

    # ============================================================
    # TRAIN
    # ============================================================
    torch.cuda.empty_cache()
    gc.collect()

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset,
        processing_class=tokenizer,
        formatting_func=formatting_prompts_func,
        callbacks=[GPUMetricsCallback(log_every_n_steps=2)],
    )

    print("Starting IGU-SFT training v9.2 ...")
    t0 = time.time()
    trainer.train()
    elapsed = time.time() - t0
    print(f"Training done in {elapsed / 60:.1f} min")

    ADAPTER_DIR = "/kaggle/working/sft_adapter"
    model.save_pretrained(ADAPTER_DIR)
    tokenizer.save_pretrained(ADAPTER_DIR)
    print(f"Adapter saved to {ADAPTER_DIR}")
else:
    print("USE_PRETRAINED=1: skipping SFT training.")

## Package TensorBoard Logs for Download

In [ ]:
if TRAIN_ON_KAGGLE:
    import os, zipfile
    from pathlib import Path

    TB_LOG_DIR  = "/kaggle/working/tb_logs"
    ZIP_OUTPUT  = "/kaggle/working/tensorboard_logs.zip"

    log_path = Path(TB_LOG_DIR)
    if log_path.exists():
        files  = [f for f in log_path.rglob("*") if f.is_file()]
        events = list(log_path.rglob("events.out.tfevents.*"))
        print(f"Packaging {len(events)} event files ({len(files)} total)")
        with zipfile.ZipFile(ZIP_OUTPUT, "w", zipfile.ZIP_DEFLATED) as zf:
            for fp in files:
                zf.write(fp, fp.relative_to(log_path.parent))
        sz = os.path.getsize(ZIP_OUTPUT) / (1024 * 1024)
        print(f"=> {ZIP_OUTPUT} ({sz:.2f} MB)")
    else:
        print(f"[WARN] No TB logs at {TB_LOG_DIR}")

## Mode B: Load Pre-trained LoRA

In [ ]:
if USE_PRETRAINED:
    import os

    SRC_ADAPTER_DIR = PRETRAINED_ADAPTER_DATASET_PATH
    required_files  = ["adapter_config.json", "adapter_model.safetensors"]

    print("Using pre-trained adapter from:", SRC_ADAPTER_DIR)
    for fname in required_files:
        fpath = os.path.join(SRC_ADAPTER_DIR, fname)
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing required adapter file: {fpath}")
        print(f"  {fname}: {os.path.getsize(fpath)/1024/1024:.1f} MB")
else:
    print("TRAIN_ON_KAGGLE=1: pretrained adapter path check skipped.")

## Create submission.zip

In [ ]:
import json, os, shutil, zipfile

OUTPUT_DIR             = "/kaggle/working"
SUBMISSION_ADAPTER_DIR = os.path.join(OUTPUT_DIR, "submission_adapter")
os.makedirs(SUBMISSION_ADAPTER_DIR, exist_ok=True)

required_files = ["adapter_config.json", "adapter_model.safetensors"]

if TRAIN_ON_KAGGLE:
    src_adapter_dir = "/kaggle/working/sft_adapter"
    print("Packaging freshly trained adapter from:", src_adapter_dir)
else:
    src_adapter_dir = PRETRAINED_ADAPTER_DATASET_PATH
    print("Packaging pre-trained adapter from:", src_adapter_dir)

for fname in required_files:
    src = os.path.join(src_adapter_dir, fname)
    dst = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
    if not os.path.exists(src):
        raise FileNotFoundError(f"Missing required adapter file: {src}")
    shutil.copy2(src, dst)
    print(f"Copied {fname} ({os.path.getsize(dst)/1024/1024:.1f} MB)")

config_path = os.path.join(SUBMISSION_ADAPTER_DIR, "adapter_config.json")
with open(config_path, "r") as f:
    cfg = json.load(f)

cfg["base_model_name_or_path"] = BASE_MODEL_NAME
cfg["inference_mode"]          = True
cfg["lora_dropout"]            = 0.0

with open(config_path, "w") as f:
    json.dump(cfg, f, indent=2)

zip_path = os.path.join(OUTPUT_DIR, "submission.zip")
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in required_files:
        fpath = os.path.join(SUBMISSION_ADAPTER_DIR, fname)
        zf.write(fpath, fname)
        print(f"  Added {fname}")

zip_sz = os.path.getsize(zip_path) / 1024 / 1024
print(f"\nsubmission.zip: {zip_sz:.1f} MB")
print("Done. Ready to submit.")